In [1]:
import pandas as pd

In [5]:
names = pd.read_csv('../data/imdb/name.basics.tsv.gz', sep='\t')
names.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


Tu mamy informacje o osobach (nie tylko aktorach, ale ogólnie wszystkich osobach związanych z filmami)
- Imie i nazwisko
- Daty urodzin / śmierci (może przydatne jedynie do filtrowania po żyjących)
- professions (tutaj po przecinku wymienionych kilka, w dalszych analizach albo to trzeba robić na wiele wierszy, albo na wiele kolumn i zrobić one-hot-encoding)
- tytuły z którymi są związani (znów ta sama sytuacja że wiele po przecinku)

In [ ]:
title_akas = pd.read_csv('../data/imdb/title.akas.tsv.gz', sep='\t')
title_akas.head()

Tutaj mamy wiele wersji tego samego filmu. Jest oryginalny tytuł i dzięki temu można wziąć region i język. Wszystkie inne duplikaty tego samego tytułu w różnych wersjach jezykowych trzeba by usunąc. Ale jest ryzykowne łączenie potem po tytule oryginalnym, bo w innych bazach będą to tytuły angielskie


In [17]:
title_basics = pd.read_csv('../data/imdb/title.basics.tsv.gz', sep='\t')
title_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [5]:
title_crew = pd.read_csv('../data/imdb/title.crew.tsv.gz', sep='\t')
title_crew.head()

,tconst,directors,writers
0,tt0000001,nm0005690,\N
1,tt0000002,nm0721526,\N
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,\N
4,tt0000005,nm0005690,\N


To jest dość ważne bo mamy od razu info o reżyserze i scenarzyście (dokładnie tego potrzebujemy do naszej analizy) i to trzeba będzie łączyć po id z tabelą names
UWAGA bo czasem jeden film ma kilku reżyserów/aktorów dlatego trzeba to uważnie połączyć

In [6]:
# Przygotowanie reżyserów do łączenia
directors_exploded = title_crew[['tconst', 'directors']].copy()
directors_exploded['directors'] = directors_exploded['directors'].str.split(',')
directors_exploded = directors_exploded.explode('directors')
directors_exploded.head(10)

,tconst,directors
0,tt0000001,nm0005690
1,tt0000002,nm0721526
2,tt0000003,nm0721526
3,tt0000004,nm0721526
4,tt0000005,nm0005690
5,tt0000006,nm0005690
6,tt0000007,nm0005690
6,tt0000007,nm0374658
7,tt0000008,nm0005690
8,tt0000009,nm0085156


In [7]:
directors_named = pd.merge(
    directors_exploded, 
    names[['nconst', 'primaryName']], 
    left_on='directors', 
    right_on='nconst', 
    how='left')

directors_named.drop('nconst', inplace=True, axis=1)

In [8]:
directors_named.head(10)

,tconst,directors,primaryName
0,tt0000001,nm0005690,William K.L. Dickson
1,tt0000002,nm0721526,Émile Reynaud
2,tt0000003,nm0721526,Émile Reynaud
3,tt0000004,nm0721526,Émile Reynaud
4,tt0000005,nm0005690,William K.L. Dickson
5,tt0000006,nm0005690,William K.L. Dickson
6,tt0000007,nm0005690,William K.L. Dickson
7,tt0000007,nm0374658,William Heise
8,tt0000008,nm0005690,William K.L. Dickson
9,tt0000009,nm0085156,Alexander Black


In [9]:
# Przygotowanie scenarzystów do łączenia
writers_exploded = title_crew[['tconst', 'writers']].copy()
writers_exploded['writers'] = writers_exploded['writers'].str.split(',')
writers_exploded = writers_exploded.explode('writers')
writers_exploded.head(10)

,tconst,writers
0,tt0000001,\N
1,tt0000002,\N
2,tt0000003,nm0721526
3,tt0000004,\N
4,tt0000005,\N
5,tt0000006,\N
6,tt0000007,\N
7,tt0000008,\N
8,tt0000009,nm0085156
9,tt0000010,\N


In [10]:
writers_named = pd.merge(
    writers_exploded, 
    names[['nconst', 'primaryName']], 
    left_on='writers', 
    right_on='nconst', 
    how='left')

writers_named.drop('nconst', inplace=True, axis=1)

In [11]:
writers_named.head(10)

,tconst,writers,primaryName
0,tt0000001,\N,NaN
1,tt0000002,\N,NaN
2,tt0000003,nm0721526,Émile Reynaud
3,tt0000004,\N,NaN
4,tt0000005,\N,NaN
5,tt0000006,\N,NaN
6,tt0000007,\N,NaN
7,tt0000008,\N,NaN
8,tt0000009,nm0085156,Alexander Black
9,tt0000010,\N,NaN


In [12]:
directors_named.to_csv("../data/merged/directors_named.csv", index = False)
writers_named.to_csv("../data/merged/writers_named.csv", index = False)

Episodes nie potrzebujemy bo to będzie do odcinków seriali itp, całkiem możemy tą tabele ominąć

In [14]:
#needed_cols = ['tconst', 'nconst', 'category'] 

#title_principals = pd.read_csv('../data/imdb/title.principals.tsv.gz', sep='\t', usecols=needed_cols)

To może być potrzebne żeby wyfiltrować np samych aktorów, ale to jest tak ogromny plik że próbując to załadować brakło mi pamięci a potem wyrzuciło mi cały kernel. Nie wiem czy wam się uda. Aktorów można też z innych plików wziąć

Nie wiem czy ratings będą nam potrzebne (bo jak mamy określać sukces filmu w przyszłości to ratingu jeszcze nie znamy). Ale może się wykorzysta np do tego jak są oceniane filmy z danym aktorem/reżyserem (?)

Tu jest fajna tego baza bo od razu mamy średnią ocene wielu użytkowników, a nie każdego użytkownika osobno jak to jest w innych plikach

In [25]:
title_basics['startYear'] = pd.to_numeric(title_basics['startYear'], errors='coerce')
title_movies = title_basics[(title_basics['startYear'] >= 1990) & (title_basics['startYear'] < 2026) & (title_basics['primaryTitle'].notna()) & (title_basics['titleType'] == 'movie')].copy()
title_movies

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
11630,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,\N,\N,"Action,Crime"
15478,tt0015724,movie,Dama de noche,Dama de noche,0,1993.0,\N,102,"Drama,Mystery,Romance"
34793,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,\N,118,"Comedy,Fantasy,Romance"
37407,tt0038086,movie,Shiva und die Galgenblume,Shiva und die Galgenblume,0,1993.0,\N,\N,Thriller
46124,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,\N,\N,\N
...,...,...,...,...,...,...,...,...,...
12347191,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015.0,\N,57,Documentary
12347215,tt9916680,movie,De la ilusión al desconcierto: cine colombiano...,De la ilusión al desconcierto: cine colombiano...,0,2007.0,\N,100,Documentary
12347227,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,\N,\N,Comedy
12347237,tt9916730,movie,6 Gunn,6 Gunn,0,2017.0,\N,116,Drama


In [26]:
title_movies.shape
# ponad 600TYS filmów

(412926, 9)

In [28]:
title_movies.to_csv("../data/merged/imdb.csv", index=False)